# Activity #1: RAGAS Evaluation for OpenAI vs FireWOrks AI Provider

## 1. Environment and dependencies

Check the .env.example and set all the key and env variables required to run in .env file.

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key:")

# Enable LangSmith tracing for token/cost analysis
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ.setdefault("LANGCHAIN_PROJECT", "activity2-agent-helpfulness-eval")

if not os.environ.get("LANGCHAIN_API_KEY"):
    try:
        key = getpass("Optional: Enter LangSmith API key (Enter to skip):")
        if key:
            os.environ["LANGCHAIN_API_KEY"] = key
    except Exception:
        pass

## 2. Imports

In [3]:
import time
import pandas as pd
import nest_asyncio
nest_asyncio.apply()  # for RAGAS async in Jupyter
from langchain_core.messages import HumanMessage

from app.graphs.agent_with_helpfulness import graph as agent_graph
from app.rag import _get_rag_graph
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    Faithfulness,
    LLMContextRecall,
    ContextEntityRecall,
    ContextPrecision,
    FactualCorrectness,
    ResponseRelevancy,
)
from langchain_openai import ChatOpenAI

## 3. Evaluation dataset

Generate the eval dataset using RAGAS **TestsetGenerator** (as in `Evaluating_RAG_Assignment.ipynb`): load source documents, set up the generator LLM and embeddings, then call `generate_with_langchain_docs`. Ensure `data/HealthWellnessGuide.txt` exists, or change the path to your document(s).

In [4]:
# Load source documents (same as Evaluating_RAG_Assignment.ipynb)
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

 # Load PDFs from data directory (recursive)
try:
    directory_loader = DirectoryLoader(
        "data", glob="**/*.pdf", loader_cls=PyMuPDFLoader
    )
    docs = directory_loader.load()
except Exception:
    docs = []

# Generator LLM and embeddings for TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

# Generate synthetic test set with RAGAS TestsetGenerator
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

eval_df = dataset.to_pandas()
eval_df

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/22 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Property 'summary' already exists in node 'e88599'. Skipping!
Property 'summary' already exists in node '935275'. Skipping!
Property 'summary' already exists in node '73240d'. Skipping!
Property 'summary' already exists in node '950263'. Skipping!


...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,As a cat owner dedicated to proactive care and...,"[their cat’s maturation and aging process, and...",Table 2 is highlighted as a comprehensive reso...,single_hop_specifc_query_synthesizer
1,what is Feline-Friendly Handling and Nursing C...,[Feline-Friendly Strategies Feline-friendly ha...,Feline-Friendly Handling and Nursing Care Guid...,single_hop_specifc_query_synthesizer
2,Wen shud I take my cat to the veternarian?,"[For example, some senior cats aged 10 years a...",Senior cats aged 10 years and older may be tre...,single_hop_specifc_query_synthesizer
3,"According to the AAFP Senior Care Guidelines, ...",[Discussion Items for All Life Stages The Task...,Senior cats should be seen at least every 6 mo...,single_hop_specifc_query_synthesizer
4,How can behavioral assessment and counseling d...,[<1-hop>\n\ndetection of changes and identiﬁca...,Behavioral assessment and counseling during ve...,multi_hop_abstract_query_synthesizer
5,How do the principles of informed consent appl...,[<1-hop>\n\nINFORMED CONSENT This work did not...,The principles of informed consent in feline v...,multi_hop_abstract_query_synthesizer
6,According to the 2021 AAHA/AAFP Feline Life St...,"[<1-hop>\n\n90. Wichert B, Muller L, Gebert S,...",The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_abstract_query_synthesizer
7,"According to veterinary clinical guidelines, w...","[<1-hop>\n\nspace, administration at this loca...",Veterinary clinical guidelines recommend sever...,multi_hop_abstract_query_synthesizer
8,Wut resors from the Centers for Disease Contro...,"[<1-hop>\n\n90. Wichert B, Muller L, Gebert S,...",The Centers for Disease Control and Prevention...,multi_hop_specific_query_synthesizer
9,How can a cat owner from Mar/Apr 2021 help pre...,[<1-hop>\n\nInvestigating Urine Marking” box. ...,"To help prevent urine marking, a cat owner sho...",multi_hop_specific_query_synthesizer


## 4. Run agent and collect responses + retrieved contexts with OPENAI as provider

For each question we:
1. Invoke `agent_with_helpfulness` (with LangSmith tracing).
2. Get retrieved contexts from the app RAG graph (same query) for RAGAS retrieval metrics.

In [6]:
def get_final_response_content(messages):
    """Extract the last non–tool, non-internal AI message content."""
    for m in reversed(messages):
        content = getattr(m, "content", "")
        if not content:
            continue
        if isinstance(content, str) and (
            content.startswith("HELPFULNESS:") or content.startswith("VIBE:")
        ):
            continue
        return content
    return ""


rag_graph = _get_rag_graph()
responses = []
retrieved_contexts_list = []

for idx, row in eval_df.iterrows():
    question = row["user_input"]
    # Run agent
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]})
    final_response = get_final_response_content(result["messages"])
    responses.append(final_response)
    # Get retrieved contexts from RAG (for RAGAS retrieval quality)
    try:
        rag_result = rag_graph.invoke({"question": question})
        ctxs = rag_result.get("context", [])
        retrieved_contexts_list.append([c.page_content for c in ctxs] if ctxs else [""])
    except Exception:
        retrieved_contexts_list.append([""])
    #time.sleep(0.5)  # gentle rate limiting

eval_df["response"] = responses
eval_df["retrieved_contexts"] = retrieved_contexts_list
eval_df

,user_input,reference_contexts,reference,synthesizer_name,response,retrieved_contexts
0,As a cat owner dedicated to proactive care and...,"[their cat’s maturation and aging process, and...",Table 2 is highlighted as a comprehensive reso...,single_hop_specifc_query_synthesizer,Table 2 is important in feline healthcare beca...,[Introduction\nThe feline patient’s life stage...
1,what is Feline-Friendly Handling and Nursing C...,[Feline-Friendly Strategies Feline-friendly ha...,Feline-Friendly Handling and Nursing Care Guid...,single_hop_specifc_query_synthesizer,Feline-Friendly Handling and Nursing Care Guid...,"[their cat’s maturation and aging process, and..."
2,Wen shud I take my cat to the veternarian?,"[For example, some senior cats aged 10 years a...",Senior cats aged 10 years and older may be tre...,single_hop_specifc_query_synthesizer,You should take your cat to the veterinarian f...,"[For example, some senior cats aged 10 years a..."
3,"According to the AAFP Senior Care Guidelines, ...",[Discussion Items for All Life Stages The Task...,Senior cats should be seen at least every 6 mo...,single_hop_specifc_query_synthesizer,"According to the AAFP Senior Care Guidelines, ...","[For example, some senior cats aged 10 years a..."
4,How can behavioral assessment and counseling d...,[<1-hop>\n\ndetection of changes and identiﬁca...,Behavioral assessment and counseling during ve...,multi_hop_abstract_query_synthesizer,Behavioral assessment and counseling during ve...,[than observed or learned.44 Important aspects...
5,How do the principles of informed consent appl...,[<1-hop>\n\nINFORMED CONSENT This work did not...,The principles of informed consent in feline v...,multi_hop_abstract_query_synthesizer,The principles of informed consent in the publ...,[INFORMED CONSENT\nThis work did not involve t...
6,According to the 2021 AAHA/AAFP Feline Life St...,"[<1-hop>\n\n90. Wichert B, Muller L, Gebert S,...",The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_abstract_query_synthesizer,The 2021 AAHA/AAFP Feline Life Stage Guideline...,[Introduction\nThe feline patient’s life stage...
7,"According to veterinary clinical guidelines, w...","[<1-hop>\n\nspace, administration at this loca...",Veterinary clinical guidelines recommend sever...,multi_hop_abstract_query_synthesizer,"According to veterinary clinical guidelines, p...","[space, administration at this location is not..."
8,Wut resors from the Centers for Disease Contro...,"[<1-hop>\n\n90. Wichert B, Muller L, Gebert S,...",The Centers for Disease Control and Prevention...,multi_hop_specific_query_synthesizer,The CDC provides resources that help cat owner...,"[space, administration at this location is not..."
9,How can a cat owner from Mar/Apr 2021 help pre...,[<1-hop>\n\nInvestigating Urine Marking” box. ...,"To help prevent urine marking, a cat owner sho...",multi_hop_specific_query_synthesizer,To help prevent both urine marking and dental ...,[behavior in a small group of 17 free-roaming ...


## 5. RAGAS evaluation with OPENAI as provider



In [8]:
from ragas import RunConfig

evaluation_dataset = EvaluationDataset.from_pandas(eval_df)

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
run_config = RunConfig(timeout=360)

ragas_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),   # retrieval quality
        ContextEntityRecall(),
        ContextPrecision(),
        Faithfulness(),        # answer faithfulness to context
        ResponseRelevancy(),   # answer relevancy to question
        FactualCorrectness(),  # end-to-end accuracy vs reference
    ],
    llm=evaluator_llm,
    run_config=run_config,
)

print("RAGAS results with OpenAI as provider:", ragas_result)
ragas_result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

RAGAS results with OpenAI as provider: {'context_recall': 0.9114, 'context_entity_recall': 0.2371, 'context_precision': 0.9861, 'faithfulness': 0.9183, 'answer_relevancy': 0.9549, 'factual_correctness': 0.4850}


{'context_recall': 0.9114, 'context_entity_recall': 0.2371, 'context_precision': 0.9861, 'faithfulness': 0.9183, 'answer_relevancy': 0.9549, 'factual_correctness': 0.4850}

## 6. LangSmith Observability with OPENAI as provider

In [ ]:
# LangSmith evaluation via langsmith.evaluation.evaluate (same eval dataset as RAGAS)
# Uses agent_with_helpfulness; results include per-example execution_time (latency).
from langsmith import Client
from langsmith.evaluation import evaluate as langsmith_evaluate
import uuid

def _run_agent_for_evaluate(example: dict):
    """Target for LangSmith evaluate: expects inputs with 'question', invokes agent, returns output."""
    question = example.get("question") or example.get("user_input", "")
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]})
    final = get_final_response_content(result["messages"])
    return {"output": final}

_langsmith_client = Client()
_ls_dataset_name = f"OPENAI Provider agent-eval-{uuid.uuid4().hex[:8]}"
_ls_dataset = _langsmith_client.create_dataset(
    dataset_name=_ls_dataset_name,
    description="Agent helpfulness eval (same as RAGAS eval_df) for LangSmith evaluate",
)
for _, _row in eval_df.iterrows():
    _langsmith_client.create_example(
        inputs={"question": _row["user_input"]},
        dataset_id=_ls_dataset.id,
    )

_langsmith_eval_results = langsmith_evaluate(
    _run_agent_for_evaluate,
    data=_ls_dataset_name,
    evaluators=[],
)
_langsmith_eval_df = _langsmith_eval_results.to_pandas()

# Latency summary from execution_time (seconds)
if "execution_time" in _langsmith_eval_df.columns:
    _et = _langsmith_eval_df["execution_time"]
    print("LangSmith evaluate — latency (execution_time, seconds):")
    print(f"  Mean: {_et.mean():.3f}  Median (P50): {_et.median():.3f}  Min: {_et.min():.3f}  Max: {_et.max():.3f}")
_langsmith_eval_df

View the evaluation results for experiment: 'stupendous-leg-10' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/cface519-7aeb-4c57-a021-af6673b2c093/compare?selectedSessions=6ce8b8f0-02e1-48bb-9822-5d753eb33c67




0it [00:00, ?it/s]

LangSmith evaluate — latency (execution_time, seconds):
  Mean: 6.817  Median (P50): 6.262  Min: 3.256  Max: 14.098


,inputs.question,outputs.output,error,execution_time,example_id,id
0,What are the key physical and behavioral consi...,The key physical and behavioral considerations...,None,6.193875,28474501-0b52-4052-818f-4282627866ec,019ccea1-1cb4-7f20-b8b6-2acd845fa6e6
1,How can a veterinarian use individualized care...,Veterinarians can use individualized care reco...,None,8.511478,dac6f4be-4992-4eb9-a255-7ba4cb7c0f54,019ccea1-34e7-7783-9ae3-1b9507a2dbbe
2,How can a cat owner from Mar/Apr 2021 help pre...,To help prevent urine marking and dental probl...,None,7.513059,f848ff48-ca57-4be3-ac19-f53caaabbf55,019ccea1-5628-7223-9d2a-e149a28e3edc
3,Wut resors from the Centers for Disease Contro...,The Centers for Disease Control and Prevention...,None,5.766677,f8646892-31a9-41f2-998b-f3a005f1fc42,019ccea1-7381-7382-870b-9bf8ebe3c0a2
4,"According to veterinary clinical guidelines, w...",Preventive healthcare measures for cats to red...,None,7.409168,6a98666f-2e97-4f29-a857-b73dd59c9531,019ccea1-8a09-70c2-9ca3-e1ca1fabcb82
5,According to the 2021 AAHA/AAFP Feline Life St...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,None,5.524003,e1c82cdb-3c6c-4147-b480-d0243b7f627f,019ccea1-a6fa-7d11-be46-96b82afc921d
6,How do the principles of informed consent appl...,The principles of informed consent in the publ...,None,14.097750,5707e153-5992-4e71-98c2-b955e1b57fce,019ccea1-bc8f-78d1-b4fa-af69f5e5d4d7
7,How can behavioral assessment and counseling d...,Behavioral assessment and counseling during ve...,None,7.338709,70e98fdf-a000-4e12-afb4-aeb70fdb0945,019ccea1-f3a1-7752-bce4-c3f34e5e2d09
8,"According to the AAFP Senior Care Guidelines, ...","According to the AAFP Senior Care Guidelines, ...",None,3.256105,dc31b0cf-92ff-45d1-8946-1d146f63c412,019ccea2-104d-7bc3-b77e-16a35c144f5d
9,Wen shud I take my cat to the veternarian?,You should take your cat to the veterinarian f...,None,5.779990,c5feb028-8a36-4cd1-bdd0-611245770261,019ccea2-1d06-7ed2-ac72-befd64412464


In [ ]:
os.environ["LLM_PROVIDER"] = "fireworks"  # switch for Fireworks run

## 7. Run agent and collect responses + retrieved contexts with FIREWORKS as provider

For each question we:
1. Invoke `agent_with_helpfulness` (with LangSmith tracing).
2. Get retrieved contexts from the app RAG graph (same query) for RAGAS retrieval metrics.

In [11]:
def get_final_response_content(messages):
    """Extract the last non–tool, non-internal AI message content."""
    for m in reversed(messages):
        content = getattr(m, "content", "")
        if not content:
            continue
        if isinstance(content, str) and (
            content.startswith("HELPFULNESS:") or content.startswith("VIBE:")
        ):
            continue
        return content
    return ""


rag_graph = _get_rag_graph()
responses = []
retrieved_contexts_list = []

for idx, row in eval_df.iterrows():
    question = row["user_input"]
    # Run agent
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]})
    final_response = get_final_response_content(result["messages"])
    responses.append(final_response)
    # Get retrieved contexts from RAG (for RAGAS retrieval quality)
    try:
        rag_result = rag_graph.invoke({"question": question})
        ctxs = rag_result.get("context", [])
        retrieved_contexts_list.append([c.page_content for c in ctxs] if ctxs else [""])
    except Exception:
        retrieved_contexts_list.append([""])
    #time.sleep(0.5)  # gentle rate limiting

eval_df["response"] = responses
eval_df["retrieved_contexts"] = retrieved_contexts_list
eval_df

,user_input,reference_contexts,reference,synthesizer_name,response,retrieved_contexts
0,As a cat owner dedicated to proactive care and...,"[their cat’s maturation and aging process, and...",Table 2 is highlighted as a comprehensive reso...,single_hop_specifc_query_synthesizer,Table 2 in the feline healthcare guidelines is...,[Introduction\nThe feline patient’s life stage...
1,what is Feline-Friendly Handling and Nursing C...,[Feline-Friendly Strategies Feline-friendly ha...,Feline-Friendly Handling and Nursing Care Guid...,single_hop_specifc_query_synthesizer,Feline-Friendly Handling and Nursing Care Guid...,"[their cat’s maturation and aging process, and..."
2,Wen shud I take my cat to the veternarian?,"[For example, some senior cats aged 10 years a...",Senior cats aged 10 years and older may be tre...,single_hop_specifc_query_synthesizer,You should take your cat to the veterinarian i...,"[For example, some senior cats aged 10 years a..."
3,"According to the AAFP Senior Care Guidelines, ...",[Discussion Items for All Life Stages The Task...,Senior cats should be seen at least every 6 mo...,single_hop_specifc_query_synthesizer,The specific examination frequency for senior ...,"[For example, some senior cats aged 10 years a..."
4,How can behavioral assessment and counseling d...,[<1-hop>\n\ndetection of changes and identiﬁca...,Behavioral assessment and counseling during ve...,multi_hop_abstract_query_synthesizer,Behavioral assessment during veterinary visits...,[than observed or learned.44 Important aspects...
5,How do the principles of informed consent appl...,[<1-hop>\n\nINFORMED CONSENT This work did not...,The principles of informed consent in feline v...,multi_hop_abstract_query_synthesizer,The principles of informed consent in veterina...,[INFORMED CONSENT\nThis work did not involve t...
6,According to the 2021 AAHA/AAFP Feline Life St...,"[<1-hop>\n\n90. Wichert B, Muller L, Gebert S,...",The 2021 AAHA/AAFP Feline Life Stage Guideline...,multi_hop_abstract_query_synthesizer,The 2021 AAHA/AAFP Feline Life Stage Guideline...,[Introduction\nThe feline patient’s life stage...
7,"According to veterinary clinical guidelines, w...","[<1-hop>\n\nspace, administration at this loca...",Veterinary clinical guidelines recommend sever...,multi_hop_abstract_query_synthesizer,Preventive healthcare measures for cats to red...,"[space, administration at this location is not..."
8,Wut resors from the Centers for Disease Contro...,"[<1-hop>\n\n90. Wichert B, Muller L, Gebert S,...",The Centers for Disease Control and Prevention...,multi_hop_specific_query_synthesizer,The CDC provides several resources to help cat...,"[space, administration at this location is not..."
9,How can a cat owner from Mar/Apr 2021 help pre...,[<1-hop>\n\nInvestigating Urine Marking” box. ...,"To help prevent urine marking, a cat owner sho...",multi_hop_specific_query_synthesizer,To help prevent urine marking and dental probl...,[behavior in a small group of 17 free-roaming ...


## 8. RAGAS evaluation with FIREWORKS as provider



In [13]:
from ragas import RunConfig

evaluation_dataset_new = EvaluationDataset.from_pandas(eval_df)

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
run_config = RunConfig(timeout=360)

ragas_result = evaluate(
    dataset=evaluation_dataset_new,
    metrics=[
        LLMContextRecall(),   # retrieval quality
        ContextEntityRecall(),
        ContextPrecision(),
        Faithfulness(),        # answer faithfulness to context
        ResponseRelevancy(),   # answer relevancy to question
        FactualCorrectness(),  # end-to-end accuracy vs reference
    ],
    llm=evaluator_llm,
    run_config=run_config,
)

print("RAGAS results with Fireworks as provider:", ragas_result)
ragas_result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

RAGAS results with Fireworks as provider: {'context_recall': 0.9114, 'context_entity_recall': 0.2562, 'context_precision': 0.9861, 'faithfulness': 0.8048, 'answer_relevancy': 0.7955, 'factual_correctness': 0.3975}


{'context_recall': 0.9114, 'context_entity_recall': 0.2562, 'context_precision': 0.9861, 'faithfulness': 0.8048, 'answer_relevancy': 0.7955, 'factual_correctness': 0.3975}

## 9. LangSmith Observability with FIREWORKS as provider

In [ ]:
# LangSmith evaluation via langsmith.evaluation.evaluate (same eval dataset as RAGAS)
# Uses agent_with_helpfulness; results include per-example execution_time (latency).
from langsmith import Client
from langsmith.evaluation import evaluate as langsmith_evaluate
import uuid

def _run_agent_for_evaluate(example: dict):
    """Target for LangSmith evaluate: expects inputs with 'question', invokes agent, returns output."""
    question = example.get("question") or example.get("user_input", "")
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]})
    final = get_final_response_content(result["messages"])
    return {"output": final}

_langsmith_client = Client()
_ls_dataset_name = f"Fireworks AI agent-eval-{uuid.uuid4().hex[:8]}"
_ls_dataset = _langsmith_client.create_dataset(
    dataset_name=_ls_dataset_name,
    description="Agent helpfulness eval (same as RAGAS eval_df) for LangSmith evaluate",
)
for _, _row in eval_df.iterrows():
    _langsmith_client.create_example(
        inputs={"question": _row["user_input"]},
        dataset_id=_ls_dataset.id,
    )

_langsmith_eval_results = langsmith_evaluate(
    _run_agent_for_evaluate,
    data=_ls_dataset_name,
    evaluators=[],
)
_langsmith_eval_df = _langsmith_eval_results.to_pandas()

# Latency summary from execution_time (seconds)
if "execution_time" in _langsmith_eval_df.columns:
    _et = _langsmith_eval_df["execution_time"]
    print("LangSmith evaluate — latency (execution_time, seconds):")
    print(f"  Mean: {_et.mean():.3f}  Median (P50): {_et.median():.3f}  Min: {_et.min():.3f}  Max: {_et.max():.3f}")
_langsmith_eval_df

View the evaluation results for experiment: 'extraneous-monkey-40' at:
https://smith.langchain.com/o/8ec58a99-449f-4f06-9fbf-946e52cbfeb5/datasets/74fb8c40-526e-484f-bef4-18a5aac902b5/compare?selectedSessions=f51dece9-9f37-48d7-936a-d129a1bef21d




0it [00:00, ?it/s]

LangSmith evaluate — latency (execution_time, seconds):
  Mean: 6.128  Median (P50): 5.529  Min: 1.638  Max: 18.301


,inputs.question,outputs.output,error,execution_time,example_id,id
0,What are the key physical and behavioral consi...,"During the kitten stage, key physical and beha...",None,7.298947,0d42c44c-d0c3-431b-8d5d-0a4c2af88e95,019cced1-702f-7b12-9689-8492ffba5342
1,How can a veterinarian use individualized care...,Veterinarians can use individualized care reco...,None,5.910504,5b14b5fd-998d-420b-86a6-3881460b3ba3,019cced1-8cb4-7631-9bc5-6319b50889ae
2,How can a cat owner from Mar/Apr 2021 help pre...,To help prevent both urine marking and dental ...,None,5.573489,e7df5f3b-d6c9-43e3-8ce8-9d96f91525f8,019cced1-a3cb-7493-87ca-3ba7a4e9814d
3,Wut resors from the Centers for Disease Contro...,The Centers for Disease Control and Prevention...,None,4.981771,85b9f2d1-f466-4552-bdf3-799aa7e29653,019cced1-b991-7be1-8c7a-899e0f76df6e
4,"According to veterinary clinical guidelines, w...","According to veterinary clinical guidelines, p...",None,6.220028,b86d1603-9d41-40bb-8e4e-d059dc472758,019cced1-cd08-7722-8ea6-122dd9ffe8ea
5,According to the 2021 AAHA/AAFP Feline Life St...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,None,6.603630,ecf6daeb-6d01-4eba-9f1a-4463404b3a7c,019cced1-e554-7523-b4cc-060afe9f8dd3
6,How do the principles of informed consent appl...,The principles of informed consent in the publ...,None,18.300878,bc3b1384-4a2b-44ab-b5a5-8010a581f9d6,019cced1-ff21-7253-b56f-dd7d1fcb9c19
7,How can behavioral assessment and counseling d...,Behavioral assessment and counseling during ve...,None,5.485078,f6fde7c6-cee1-492d-8e86-a65bfd1061e6,019cced2-469f-7870-9503-331d14db4391
8,"According to the AAFP Senior Care Guidelines, ...","According to the AAFP Senior Care Guidelines, ...",None,2.859064,3e560cc2-4311-4547-805f-b3bc52bb3c2d,019cced2-5c0c-7423-af52-aa7c820e5b94
9,Wen shud I take my cat to the veternarian?,You should take your cat to the veterinarian i...,None,1.637586,408380e1-6ff6-497a-bea1-7cda7c12c7ee,019cced2-6738-7c22-b18a-624ecc9c0940


## 10. Analysis and summary

|Metric|OpenAI|Fireworks AI|
|------|------|------------|
|context_recall|9114|9114|
|context_entity_recall|0.2371|0.2562|
|context_precision|0.9861|0.9861|
|faithfulness|0.9183|0.8048|
|answer_relevancy|0.9549|0.7955|
|factual_correctness|0.4850|0.3975|

LangSmith Latency & Cost metrics

|Metric|OpenAI|Fireworks AI|
|------|------|------------|
|Latency (P50)|6.26|5.53|
|Input Token|101,035|93,235|
|Output Tokens|6505|6052|
|Total Tokens|107,540|99,287|
|Input Cost|$0.0071|$0.0067|
|Output Cost|$0.0027|$0.0025|
|Total Cost|$0.0097|$0.0091|

LangSmith OpenAI Provider
![LangSmith OpenAI Provider](images/langsmitih_openai.png)


LangSmith FireworksAI Provider
![LangSmith FireworksAI Provider](images/langsmitih_fireworksai.png)
